# Linear Regression Using R Datasets
## Solution Notebook — Extended Project

This notebook contains complete, runnable solutions, alternate implementations, extra practice answers, a full Monte-Carlo simulation, and example audience narratives. Use it to check your work after attempting the Practice Skeleton.


---
## 0. Setup


In [ ]:
library(datasets)
library(ggplot2)
# install.packages("broom")  # optional
# library(broom)

theme_set(theme_minimal(base_size = 12))


### Cheat-sheet (identical to the one embedded in the skeleton)
- `library(help = "datasets")`
- `lm(y ~ x, data = train)` / `lm(y ~ x1 + x2, data = train)`
- `sigma(model)` — RSE
- `summary(model)$r.squared`
- `predict` / `residuals` / `fitted`
- `geom_smooth(method = "lm")` + LOESS overlay


---
## 1. Investigating the R Datasets Package


In [ ]:
library(datasets)
# ?datasets          # opens help
library(help = "datasets")   # full list in a new window / viewer

head(datasets::ToothGrowth)
# ?ToothGrowth
# supp = supplement type (VC = ascorbic acid, OJ = orange juice)

# Good linear-regression candidates (examples):
# cars     — dist ~ speed
# state.x77 — Income ~ Illiteracy (or multi)
# mtcars   — mpg ~ wt + hp
# faithful — eruptions ~ waiting
# iris     — Petal.Length ~ Sepal.Length (within species)


---
## 2. Confirming Data Assumptions


In [ ]:
# 2.1 Scatter plots
ggplot(cars, aes(speed, dist)) +
  geom_point(alpha = 0.7) +
  labs(title = "cars: roughly linear")

state_df <- as.data.frame(state.x77)
ggplot(state_df, aes(Illiteracy, Income)) +
  geom_point(alpha = 0.7) +
  labs(title = "state.x77: weaker linear signal")

ggplot(pressure, aes(temperature, pressure)) +
  geom_point(alpha = 0.7) +
  labs(title = "pressure: clearly non-linear — reject for plain LR")


In [ ]:
# 2.2 Correlations
cor.test(cars$speed, cars$dist)          # strong positive
cor.test(state_df$Illiteracy, state_df$Income)  # moderate negative


In [ ]:
# 2.3 Boxplots
ggplot(cars, aes(x = "", y = speed)) + geom_boxplot() + labs(title = "cars$speed")
ggplot(state_df, aes(x = "", y = Illiteracy)) + geom_boxplot() + labs(title = "Illiteracy")


---
## 3. Making and Assessing the Model


In [ ]:
set.seed(123)

# cars 60/40 split
cars_sample <- sample(c(TRUE, FALSE), nrow(cars), replace = TRUE, prob = c(0.6, 0.4))
cars_train  <- cars[cars_sample, ]
cars_test   <- cars[!cars_sample, ]

# state split
state_sample <- sample(c(TRUE, FALSE), nrow(state_df), replace = TRUE, prob = c(0.6, 0.4))
state_train  <- state_df[state_sample, ]
state_test   <- state_df[!state_sample, ]

nrow(cars_train); nrow(cars_test)


In [ ]:
# 3.2 Fit
car_model   <- lm(dist ~ speed, data = cars_train)
state_model <- lm(Income ~ Illiteracy, data = state_train)


In [ ]:
# 3.3 RSE
sigma(car_model)    # ~15.35 ft  — typical residual size in feet
sigma(state_model)  # ~627.6 dollars


In [ ]:
# 3.4 R²
summary(car_model)$r.squared    # ~0.71  → ~71 % of variance in stopping distance explained by speed
summary(state_model)$r.squared  # ~0.20


In [ ]:
# 3.5 Full summaries
summary(car_model)   # speed *** (p < 0.001)
summary(state_model) # Illiteracy *  (p ~ 0.014)


---
## 4. Residuals & LOESS


In [ ]:
# 4.1 Residual segments — cars
cars_train$estimate  <- predict(car_model)
cars_train$residuals <- residuals(car_model)

ggplot(cars_train, aes(speed, dist)) +
  geom_point(alpha = 0.6) +
  geom_point(aes(y = estimate), color = "blue", size = 2) +
  geom_segment(aes(xend = speed, yend = estimate), color = "gray70") +
  labs(title = "Actual (black) vs fitted (blue) with residual segments")

# state
state_train$estimate  <- predict(state_model)
state_train$residuals <- residuals(state_model)

ggplot(state_train, aes(Illiteracy, Income)) +
  geom_point(alpha = 0.6) +
  geom_point(aes(y = estimate), color = "blue", size = 2) +
  geom_segment(aes(xend = Illiteracy, yend = estimate), color = "gray70")


In [ ]:
# 4.2 OLS + LOESS
ggplot(cars_train, aes(speed, dist)) +
  geom_point() +
  geom_smooth(method = "lm", se = TRUE, color = "blue") +
  geom_smooth(method = "loess", se = FALSE, color = "red") +
  labs(title = "cars training: OLS (blue) tracks LOESS (red) well — high R²")

ggplot(state_train, aes(Illiteracy, Income)) +
  geom_point() +
  geom_smooth(method = "lm", se = TRUE, color = "blue") +
  geom_smooth(method = "loess", se = FALSE, color = "red") +
  labs(title = "state: more scatter, LOESS still roughly linear but weaker signal")


---
## 5. Multiple Linear Regression


In [ ]:
head(state_train)

# First try Illiteracy + HS Grad
state_m1 <- lm(Income ~ Illiteracy + `HS Grad`, data = state_train)
summary(state_m1)
# R² jumps; Illiteracy often loses significance

# Drop Illiteracy, add Population
state_m2 <- lm(Income ~ `HS Grad` + Population, data = state_train)
summary(state_m2)
# Population borderline; HS Grad remains useful


---
## 6. Alternate Code Paths


In [ ]:
# 6.1 Base-R plot + abline
plot(dist ~ speed, data = cars_train, main = "Base-R scatter + OLS line")
abline(car_model, col = "red", lwd = 2)

# 6.2 Matrix OLS (manual)
X <- cbind(1, cars_train$speed)
y <- cars_train$dist
beta_hat <- solve(t(X) %*% X) %*% t(X) %*% y
beta_hat   # should match coef(car_model)

# 6.3 broom (if installed)
# broom::tidy(car_model)
# broom::glance(car_model)


---
## 7. More Practice — Solutions


In [ ]:
# 7.1 mtcars
mt_simple <- lm(mpg ~ wt, data = mtcars)
mt_multi  <- lm(mpg ~ wt + hp, data = mtcars)
summary(mt_simple)$adj.r.squared
summary(mt_multi)$adj.r.squared   # usually improves

# 7.2 pressure — try log
pressure$log_pressure <- log(pressure$pressure)
press_lm <- lm(log_pressure ~ temperature, data = pressure)
summary(press_lm)   # much higher R² after log transform

# 7.3 Prediction interval at 21 mph
predict(car_model, newdata = data.frame(speed = 21), interval = "prediction", level = 0.95)


---
## 8. Monte-Carlo Simulation (full working version)


In [ ]:
# ---- PARAMETERS (change freely) ----
n_obs          <- 50
noise_sd       <- 15
true_intercept <- -17.6
true_slope     <- 3.93
n_sims         <- 300
set.seed(42)

r2_vec    <- numeric(n_sims)
slope_vec <- numeric(n_sims)
cover_vec <- logical(n_sims)   # does 95 % CI contain true_slope?

for (i in seq_len(n_sims)) {
  x <- runif(n_obs, 4, 25)
  y <- true_intercept + true_slope * x + rnorm(n_obs, 0, noise_sd)
  mod <- lm(y ~ x)
  r2_vec[i]    <- summary(mod)$r.squared
  slope_vec[i] <- coef(mod)[2]
  ci <- confint(mod, "x", level = 0.95)
  cover_vec[i] <- (ci[1] <= true_slope) && (true_slope <= ci[2])
}

# Visualise
par(mfrow = c(1, 2))
hist(r2_vec, breaks = 20, col = "steelblue",
     main = sprintf("R² distribution\n(mean = %.2f)", mean(r2_vec)),
     xlab = "R²")
abline(v = mean(r2_vec), col = "red", lwd = 2)

hist(slope_vec, breaks = 20, col = "darkorange",
     main = sprintf("Slope distribution\n(mean = %.2f)", mean(slope_vec)),
     xlab = "Estimated slope")
abline(v = true_slope, col = "red", lwd = 2)
abline(v = mean(slope_vec), col = "blue", lty = 2)

cat("Empirical 95 % CI coverage for slope:", mean(cover_vec), "\n")
cat("Mean R²:", mean(r2_vec), "  SD(R²):", sd(r2_vec), "\n")


### What to try next in the simulation
- Raise `noise_sd` to 30 → R² distribution shifts left and widens.
- Increase `n_obs` to 200 → R² concentrates and coverage stays near 95 %.
- Set `true_slope = 0` → most R² near 0, coverage still ~95 % (Type-I behaviour).


---
## 9. Example Audience Narratives


In [ ]:
cat("=== ANALYST / TECHNICIAN ===\n")
cat("On the training split of the classic cars data, lm(dist ~ speed) yields RSE ≈ 15.4 ft and R² ≈ 0.71 (slope p < 0.001). Residual-versus-fitted and the LOESS overlay show no material non-linearity. The state.x77 single-predictor model is weak (R² ≈ 0.20); adding HS Grad renders Illiteracy non-significant, illustrating confounding. Monte-Carlo draws with σ = 15 recover nominal 95 % CI coverage for the slope and an R² distribution centred near 0.65–0.75 for n = 50.\n\n")

cat("=== EXECUTIVE ===\n")
cat("Stopping distance rises by roughly four feet for every extra mile-per-hour of speed; speed alone accounts for most of the variation we observe. Typical prediction error is about 15 feet. We are highly confident the relationship is real. Once high-school graduation rates are taken into account, illiteracy no longer appears to drive state income—education is the clearer lever. Recommendation: prioritise further measurement of education-related predictors before acting on illiteracy alone.\n\n")

cat("=== NON-SPECIALIST LEARNER ===\n")
cat("Imagine a cloud of points showing how fast a car was going and how long it took to stop. A straight line drawn through that cloud captures the main pattern: faster cars need more distance. The number we call R² tells us that about 70 % of the differences in stopping distance can be linked to speed. The leftover gaps between each point and the line are the residuals—the part the simple line cannot explain. When we add a little random noise and repeat the experiment many times, the R² values stay in a similar range, which gives us confidence the pattern is not a fluke.\n")


---
## Review & Next Steps

You have now completed the full extended pipeline:
1. Dataset inventory → assumption checks → train/test split
2. Simple lm → RSE / R² / p-values
3. Residual diagnostics + LOESS
4. Multiple regression & confounding illustration
5. Alternate base-R / matrix / broom routes
6. Extra practice (mtcars, transforms, prediction intervals)
7. Parameterised Monte-Carlo simulation of R² and slope behaviour
8. Audience-adapted communication

**Suggested follow-on:** pick any new data.frame (UCI, your own CSV, or another built-in set), copy the reusable template structure from the Strategy Guide, and produce a short 1-page summary for a chosen audience.
